# Modellierungsseminar Sommer 2026
## Cycle Planning for workforce scheduling

In [ ]:
# import required packages
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from dataclasses import dataclass
import src.Shift as Shift # tailor-made data type for shift definitions

### inputs and parameters

In [ ]:

# basic inputs and parameters

Weekdays = range(1,8)  # results in 1,...,7 => let 1 be Monday and 7 be Sunday
Shifts = ["frueh", "spaet", "nacht", "frei"] # including free shifts
WorkShifts = ["frueh", "spaet", "nacht"] # excluding free shifts

# improvements outstanding:
    # use files for parameter input
        # shift definitions
        # available staff
        # user objectives: weighted priorities
    

# more relevant for more complex models going forward
MAX_CYCLE_WEEKS = int(52/4)  # in future this shall be user's input => too long snakes do not help to ensure "fair" distribution of shifts
DICT_WEEKDAYS = {'Mon':1,'Tue':2,'Wed':3,'Thu':4,'Fri':5,'Sat':6,'Sun':7,
                 'Monday':1,'Tuesday':2,'Wednesday':3,'Thursday':4,'Friday':5,'Saturday':6,'Sunday':7,
                 'Mo':1,'Tu':2,'We':3,'Th':4,'Fr':5,'Sa':6,'Su':7,
                 '1':1,'2':2,'3':3,'4':4,'5':5,'6':6,'7':7

}


In [ ]:
# read input data
# shift set

def readShiftSet(filename: str, mySep: str=";") -> pd.DataFrame:
    input_data = pd.read_csv(filename, sep=mySep, dtype=str) # import all values as string as first step 
    return input_data
    # potentially add data cleaning steps

folderpath = "input/"
filename = "input_ShiftDataSet.csv"

data_shiftSet = readShiftSet(folderpath + filename)
data_shiftSet["isWorkShift"] = data_shiftSet["isWorkShift"].astype(int).astype(bool)
#print(data_shiftSet)
Shifts = list(data_shiftSet["shift_ID"]) # including free shifts
WorkShifts = list(data_shiftSet[data_shiftSet["isWorkShift"]]["shift_ID"]) # excluding free shifts

#print(Shifts)
#print(data_shiftSet["shift_weekdays"])



### modelling

In [ ]:
# modelling

m = gp.Model("SnakeBuilding_simple")

# variables:
# x[s, d, sh] = 1, when snake s is working in shift sh on day d
x = m.addVars(MAX_CYCLE_WEEKS, Weekdays, Shifts, vtype=GRB.BINARY, name="x")

# active[s] = 1, when snake s is used
active = m.addVars(MAX_CYCLE_WEEKS, vtype=GRB.BINARY, name="active") # all other snakes are used as placeholders but not necessarily get activated


### conditions:

#### condition c01
_(idea is to use a unique ID for each condition for better reference)_

each shift has to be covered on each day

$$\sum_{s=1}^{n}{x_{s,d,w}} >= 1    \forall d \in D, \forall w \in W$$

$x_{s,d,w} = 1$, when snake s is working in work shift w on day d

$x: $ binary variable, 
$s: $ snake number, 
$d: $ weekday, 
$ws: $ work shift


In [ ]:
# SUBJECT TO:

# 1. each shift has to be covered on each day
for d in Weekdays:
    for ws in WorkShifts:
        m.addConstr(gp.quicksum(x[s, d, ws] for s in range(MAX_CYCLE_WEEKS)) >= 1,
                    name=f"Cover_day{d}_{ws}")


####

#### condition c02

In [ ]:
# 2. each snake can have at most one shift per day
for s in range(MAX_CYCLE_WEEKS):
    for d in Weekdays:
        m.addConstr(gp.quicksum(x[s, d, sh] for sh in Shifts) == active[s],
                    name=f"OneShiftPerDay_s{s}_d{d}")


#### condition c03

In [ ]:

# 3) at max 5 consecutive working days (ensure time for resting)
for s in range(MAX_CYCLE_WEEKS):
    for start in range(1, 7-5+1):  
        m.addConstr(
            gp.quicksum(x[s, d, sh] for d in range(start, start + 6)
                        for sh in WorkShifts) <= 5,
            name=f"Max5Work_s{s}_start{start}"
        )



### objective

In [ ]:
# set objective function: minimize number of active snakes
m.setObjective(gp.quicksum(active[s] for s in range(MAX_CYCLE_WEEKS)), GRB.MINIMIZE)

# improvements outstanding:
    # add various weighted objectives

#run optimizer
m.optimize()



### results

In [ ]:
# output (raw version, to be improved for better readability)
    # improvements outstanding: 
        # write results in file
        # create a shift overview per staff member

if m.status == GRB.OPTIMAL:
    print("\nminimum number of cycle weeks:", int(m.objVal))
    for s in range(MAX_CYCLE_WEEKS):
        if active[s].X == 1:
            print(f"\ncycle week {s+1}:")
            for d in Weekdays:
                for sh in Shifts:
                    if x[s, d, sh].X == 1:
                        print(f"  day {d}: {sh}")
